# 🎨 Graphic tools

**Time:** ~25 minutes &nbsp;•&nbsp; **Goal:** turn a plot into a figure. Same data, same four plot types — but now every mark on the canvas is there because you put it there.

A plot is what you draw to see something yourself. A figure is what you draw so somebody else sees it without you standing next to them explaining. The difference is entirely in this notebook.

## Run this first

In [ ]:
# Run this first. Every exercise below uses `df`.
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab07/data/"

DEPMAP = "depmap.csv"
if not Path(DEPMAP).exists():
    urllib.request.urlretrieve(DATA_URL + DEPMAP, DEPMAP)

df = pd.read_csv(DEPMAP)

GENES = ["SOX10", "BRAF", "KRAS", "NRAS", "PAX8", "MYB", "CTNNB1", "EGFR",
         "MDM2", "ESR1", "RAN", "RPL5", "RPS6", "PSMA1", "SNRPD1", "EIF4A3",
         "A1BG", "AADAC"]

print(df.shape)
df[["cell_line_name", "oncotree_lineage", "SOX10", "BRAF", "RAN"]].head()

## Skill 1 — Colour is information, not decoration

There are three kinds of colour scheme, and using the wrong one is a factual error rather than a stylistic one.

| scheme | for | example |
|---|---|---|
| **qualitative** | unordered categories | lineages — skin, lung, bowel |
| **sequential** | one direction, low to high | number of cell lines |
| **diverging** | a meaningful midpoint | dependency score, where **0 means nothing happened** |

A dependency score is diverging. Zero is not "low", it is "no effect", and −2 and +2 mean opposite things. A sequential ramp would hide that; a rainbow would invent an ordering the data does not have.

matplotlib's built-in maps are listed at <https://matplotlib.org/stable/users/explain/colors/colormaps.html>. `"RdBu_r"` and `"coolwarm"` are diverging; `"viridis"` is sequential and perceptually uniform; `"jet"` is the rainbow you should not use.

In [ ]:
# DEMO - a diverging map, centred properly
fig, ax = plt.subplots()
fig.set_size_inches(6, 4.5)

sc = ax.scatter(df["BRAF"], df["SOX10"],
                c=df["MDM2"], cmap="RdBu_r", vmin=-1.5, vmax=1.5,
                s=14, alpha=0.85)

ax.axhline(0, color="#9ca3af", linewidth=0.8)
ax.axvline(0, color="#9ca3af", linewidth=0.8)

bar = fig.colorbar(sc, ax=ax)
bar.set_label("MDM2 dependency")

ax.set_xlabel("BRAF dependency")
ax.set_ylabel("SOX10 dependency")
plt.show()

Two things are doing the work. `c=` takes a **column**, not a colour, and `cmap=` says how to turn those numbers into colours. And `vmin`/`vmax` are set symmetrically about zero — without them matplotlib centres the map on the data's own middle, which for a diverging scheme silently moves "white" to somewhere that means nothing.

`fig.colorbar(...)` is not optional. A colour axis without a key is decoration.

### 🎨 DIY challenge

1. Redraw it with `cmap="viridis"`. What has been lost?
2. Redraw with `cmap="RdBu_r"` but no `vmin`/`vmax`. Where does white end up, and why is that misleading?
3. Colour the points by `RAN` instead. Does a diverging map still make sense for a gene every cell needs?
4. Colour by lineage instead — a qualitative variable. You cannot pass text to `c=`, so go back to the two-`scatter`-calls trick from notebook 1, and do it for four lineages.

> 💡 Question 4 is the general rule: `c=` with a `cmap` is for **continuous** third variables. Categorical ones need one call per category.

In [ ]:
# TODO - your turn

## Skill 2 — Annotation: say the thing on the figure

A caption is read after the figure. A label on the figure is read *with* it. The three tools:

```python
ax.annotate("LN-229", xy=(x, y), xytext=(x+0.3, y+0.3),
            arrowprops=dict(arrowstyle="->"))
ax.axhline(-0.5, linestyle="--")     # a threshold
ax.text(0.5, 0.9, "n = 1,165", transform=ax.transAxes)
```

`ax.text` with `transform=ax.transAxes` is the useful one to remember: coordinates become 0–1 across the axes rather than data units, so `(0.02, 0.95)` is the top-left corner whatever the data does.

In [ ]:
# DEMO - point at the outliers and name them
fig, ax = plt.subplots()
fig.set_size_inches(6.5, 5)

ax.scatter(df["BRAF"], df["SOX10"], s=10, alpha=0.3, color="#9ca3af")

# the five most SOX10-dependent lines in the whole panel
for _, row in df.nsmallest(5, "SOX10").iterrows():
    ax.scatter(row["BRAF"], row["SOX10"], s=30, color="#5b9df9", zorder=3)
    ax.annotate(row["cell_line_name"],
                xy=(row["BRAF"], row["SOX10"]),
                xytext=(row["BRAF"] + 0.35, row["SOX10"] - 0.12),
                fontsize=9,
                arrowprops=dict(arrowstyle="->", color="#5b9df9", lw=0.8))

ax.axhline(-0.5, color="#d6a44c", linestyle="--", linewidth=1)
ax.text(0.02, 0.06, "dashed line: SOX10 = −0.5", transform=ax.transAxes,
        fontsize=9, color="#a16207")

ax.set_xlabel("BRAF dependency")
ax.set_ylabel("SOX10 dependency")
ax.set_title("The most SOX10-dependent lines are mostly, not only, melanoma")
plt.show()

Three of the five named lines are skin — and the two that are not are the interesting part. `LN-229` is a glioblastoma, and it is the single **most** SOX10-dependent line in the whole panel; `COLO 824` is filed under breast.

That is the value of annotating: the exceptions are where the next experiment comes from, and a boxplot of lineage means would never have shown you they existed. `zorder=3` keeps the highlighted points on top of the grey cloud.

### 🎨 DIY challenge

1. Do the same for `PAX8`: scatter it against `SOX10`, and annotate the five most PAX8-dependent lines. What lineage are they?
2. Add a second threshold line at `SOX10 = -2` in a different colour, and label it with `ax.text`.
3. Annotate with the **lineage** instead of the cell line name. Which is more informative here?
4. `ax.annotate` also takes `bbox=dict(boxstyle="round", fc="white", alpha=0.8)`. Add it — does it help or clutter?

> 💡 `df.nsmallest(5, "PAX8").iterrows()` gives you `index, row` pairs. `row["column_name"]` reads a value out.

In [ ]:
# TODO - your turn

## Skill 3 — Spines, ticks and limits

The default matplotlib figure has a box round it, ticks on every side, and whatever axis range the data happened to need. All three are adjustable, and all three are worth adjusting.

```python
ax.spines["top"].set_visible(False)      # drop the box
ax.spines["right"].set_visible(False)
ax.set_xlim(-4, 1)                       # a range you chose
ax.set_xticks([-3, -2, -1, 0])           # ticks where they mean something
ax.grid(axis="x", alpha=0.3)             # a grid you can read across
ax.tick_params(labelsize=9)
```

The top and right spines carry no information. Removing them is the single cheapest improvement to a matplotlib default.

In [ ]:
# DEMO - the same bar chart, before and after
top = df["oncotree_lineage"].value_counts().head(8)

fig, axes = plt.subplots(1, 2)
fig.set_size_inches(12, 3.5)

axes[0].barh(top.index[::-1], top.values[::-1], color="#5fb37a")
axes[0].set_title("default")

ax = axes[1]
ax.barh(top.index[::-1], top.values[::-1], color="#5fb37a")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
ax.set_xlabel("cell lines")
ax.tick_params(labelsize=9)
ax.set_title("tidied")

fig.tight_layout()
plt.show()

`ax.set_axisbelow(True)` puts the grid behind the bars. Without it the gridlines are drawn on top and stripe straight across your data, which looks like a mistake because it is one.

### 🎨 DIY challenge

1. Take your SOX10 histogram from notebook 1 and tidy it: no top or right spine, a horizontal grid behind the bars, and an x-range you chose.
2. Set the x-ticks explicitly to `[-3, -2, -1, 0]` and see what happens to the ones you removed.
3. `ax.set_ylim()` on a bar chart — try starting the y-axis at something other than zero, and write one sentence about why that is dishonest for bars and fine for a line.

> 💡 Question 3 is a real rule: a bar's *length* encodes the value, so a truncated axis lies. A line plot encodes position, so it does not.

In [ ]:
# TODO - your turn

## Skill 4 — A house style with `rcParams`

Every setting in the last skill applies to one `ax`. `plt.rcParams` sets the default for **every figure you draw afterwards**, so your whole document comes out consistent and you stop retyping.

```python
plt.rcParams.update({
    "figure.figsize": (6, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
```

`plt.style.use("seaborn-v0_8-whitegrid")` applies a whole pre-built set at once. `plt.style.available` lists them.

In [ ]:
# DEMO - set it once, get it everywhere
plt.rcParams.update({
    "figure.figsize": (6.5, 3.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.prop_cycle": plt.cycler(color=["#5b9df9", "#5fb37a", "#d6a44c", "#f87171"]),
})

# no styling lines at all in the plotting code now
fig, ax = plt.subplots()
ax.hist(df["SOX10"], bins=40)
ax.set_xlabel("SOX10 dependency")
ax.set_ylabel("cell lines")
ax.set_title("Same code as notebook 1, different house")
plt.show()

`axes.prop_cycle` is the list of colours matplotlib works through when you do not name one. Set it once and every unlabelled plot in your document is on-brand.

### 🎨 DIY challenge

1. Print `plt.style.available`, then try three of them on the same histogram.
2. Build your own `rcParams` block and redraw the boxplot from notebook 1 with no per-plot styling.
3. `plt.rcParams.update(plt.rcParamsDefault)` puts everything back. Check that it does.

> 💡 Put your `rcParams` block in the first cell of a notebook, right after the imports. That is where the next person will look for it.

In [ ]:
# TODO - your turn

## Skill 5 — 🏆 Getting it out of the notebook

A figure that only exists in a notebook cell is not finished. `fig.savefig()` writes it to a file, and three arguments matter:

```python
fig.savefig("sox10.png", dpi=300, bbox_inches="tight")
fig.savefig("sox10.svg", bbox_inches="tight")
```

- **`dpi`** — dots per inch. Screen is 100; print and most journals want 300 or more. A default-dpi figure in a thesis looks blurry, and that is this argument.
- **`bbox_inches="tight"`** — crops the whitespace and, more importantly, stops long axis labels being cut off at the edge.
- **the extension chooses the format.** `.png` is pixels — fine for photographs and scatter plots with thousands of points. `.svg` and `.pdf` are **vector**: instructions rather than pixels, so they stay sharp at any zoom and can be edited afterwards in Inkscape or Illustrator.

For a figure going into a paper or a thesis, save vector unless you have a reason not to.

In [ ]:
# DEMO - write it out, both ways, and check
fig, ax = plt.subplots()
ax.hist(df["SOX10"], bins=40)
ax.set_xlabel("SOX10 dependency score")
ax.set_ylabel("number of cell lines")
ax.set_title("SOX10 dependency across 1,165 cell lines")

fig.savefig("sox10.png", dpi=300, bbox_inches="tight")
fig.savefig("sox10.svg", bbox_inches="tight")
plt.show()

from pathlib import Path
for name in ["sox10.png", "sox10.svg"]:
    print(f"{name:12} {Path(name).stat().st_size / 1024:8.1f} kB")

In Colab the files land in the session's file browser — the folder icon in the left sidebar — and vanish when the session ends. Download anything you want to keep.

### 🏆 DIY finale — one figure, finished

Build a single figure that would survive being put in front of a supervisor. Pick one:

- SOX10 across the eight largest lineages, as a boxplot, with the melanoma group highlighted and a zero line
- the BRAF/SOX10 scatter, skin coloured, the three most extreme lines annotated by name
- the top ten lineages as a bar chart, with the ones under 50 cell lines greyed out

Whichever you pick, it must have:

1. axis labels **with units** — "SOX10 dependency score", not "SOX10"
2. a title that states the finding, not the variables
3. no top or right spine, and a grid behind the data if it has one
4. `n` visible somewhere — in the title, a label, or an `ax.text`
5. a deliberate colour choice you could defend in one sentence
6. saved to disk with `dpi=300` **and** as an `.svg`

Then write two sentences: what the figure shows, and one reason it might mislead.

> 💡 The second sentence is the hard one. Every figure in this notebook has an answer to it.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Choosing colormaps, including why `jet` is bad: <https://matplotlib.org/stable/users/explain/colors/colormaps.html>
- Annotation, in full: <https://matplotlib.org/stable/users/explain/text/annotations.html>
- Customising with style sheets and rcParams: <https://matplotlib.org/stable/users/explain/customizing.html>
- ColorBrewer, for schemes designed by a cartographer rather than a programmer: <https://colorbrewer2.org>

## Recap

You can now:

- pick qualitative, sequential or diverging colour, and centre a diverging map with `vmin`/`vmax`
- add a colourbar, and know that a colour axis without one is decoration
- annotate points, draw threshold lines, and place text in axes coordinates
- remove spines, control ticks and limits, and put the grid **behind** the data
- set a house style once with `rcParams` instead of retyping it
- save at `dpi=300` with `bbox_inches="tight"`, and choose vector over raster for print

**Next:** `03_multi_panel_figures` — several axes in one frame, and making them argue together.